# 결정 구조를 바꾸면 경계가 움직이는가

오답의 3분의 1이 `견적반영`↔`계약·질의검토` 경계 하나에서 나옵니다. 그리고 그 숫자가
**아무것도 바꿔도 안 움직입니다** — 파인튜닝으로 모델 계열을 바꾸고(6회), 용량을 세 배
키우고, 입력을 마스킹하고, 앙상블로 묶어도 경계 혼동은 98건에서 97건이 됐을 뿐입니다.

그래서 남은 의심이 하나 있었습니다. **분류기가 세 라벨을 한 번에 가르는 구조 자체**가
문제 아닌가? 세 라벨을 동시에 맞추려다 보면 어려운 경계에 쓸 용량이 모자랄 수 있으니까요.

이 노트북은 그 의심을 네 가지로 나눠 검사합니다.

| 구성 | 어떻게 가르는가 | 경계에 대해 |
|---|---|---|
| `multinomial` (현행) | softmax 하나로 세 라벨 동시에 | 전담 없음 |
| `OvR` | 라벨마다 "이것 대 나머지" | 전담 없음 |
| `OvO` | 세 짝마다 하나씩 | **`견적 대 계약` 전담 분류기가 생김** |
| 캐스케이드 | 1단계 `통상` 대 `검토필요` → 2단계 `견적` 대 `계약` | **2단계가 경계만 봄** |

`OvO`와 캐스케이드가 핵심입니다. 둘 다 **경계만 보는 분류기를 실제로 만들어줍니다.**
구조가 문제였다면 여기서 경계 혼동이 줄어야 합니다.

### 함께 재는 것 — 오라클 상한

`boundary_ceiling`은 경계 두 라벨만 남기고 학습·평가합니다. 진짜 경계 건만 골라
넣어주므로 **실제로는 얻을 수 없는 값**이고, 캐스케이드가 왜 그 값에 못 미치는지를
설명하는 상한입니다. 다수 클래스만 찍는 기준선과 나란히 봐야 뜻이 있습니다.

### 읽을 때 주의

fold·전처리·클래스 가중치·평가 지표는 `baselines`의 것을 그대로 씁니다. 바뀌는 것은
파이프라인 마지막 단계뿐이라 기존 기준선과 바로 견줄 수 있습니다. 평가 대상은 전체
1,024건이 아니라 **동결 앵커를 제외한 924건**입니다.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / 'scripts').is_dir()), Path.cwd().resolve())
sys.path.insert(0, str(ROOT))

from scripts.evaluation.baselines import CHAR_BALANCED, WORD_CHAR_BALANCED
from scripts.evaluation.decision_structure import boundary_ceiling, result_rows, run_all
from scripts.labeling.label_dataset import load_label_dataset

rows, meta = load_label_dataset()
print(f"{meta.get('dataset_version', '?')} · {len(rows)}건")

# 네 구성을 같은 fold로 돌린다. 10 fold x 4 구성이라 몇 분 걸린다.
table = pd.DataFrame(result_rows(run_all(rows, WORD_CHAR_BALANCED))).set_index('구성')
display(table.style.format({'fold평균 macro F1': '{:.4f}', '검토 recall': '{:.4f}',
                            '경계 비중': '{:.0%}'}))


In [ ]:
# 경계 두 라벨만 남기고 학습하면 어디까지 가는가 (오라클 라우팅)
ceiling = pd.DataFrame([
    {'입력': name, **boundary_ceiling(rows, spec)}
    for name, spec in (('word+char', WORD_CHAR_BALANCED), ('char', CHAR_BALANCED))
]).set_index('입력')
display(ceiling.style.format({'majority_baseline': '{:.3f}', 'fold_mean_accuracy': '{:.3f}',
                              'pooled_accuracy': '{:.3f}', 'pooled_macro_f1': '{:.3f}'}))


## 결론

**넷 다 현행보다 나쁩니다.** word+char 기준으로:

| 구성 | fold평균 macro F1 | 검토 recall | 경계 혼동 |
|---|---:|---:|---:|
| **multinomial (현행)** | **0.6144** | **0.5679** | 98 |
| OvR | 0.6137 | 0.5239 | 97 |
| OvO | 0.6130 | 0.5282 | 102 |
| 캐스케이드 | 0.5957 | 0.5064 | 106 |

점수 차이보다 중요한 것은 **경계 혼동이 87~106 사이에서 안 움직인다**는 사실입니다.
`OvO`는 `견적 대 계약`만 보는 분류기를 실제로 만들어주고, 캐스케이드는 `통상수용`을
아예 치워줍니다. 그런데도 경계는 그대로입니다.

**그러므로 이 경계는 결정 구조의 문제가 아닙니다.** 파인튜닝·용량·입력 변형이 못
움직였던 것과 같은 결론이고, 이번에는 "구조를 바꿔도 안 된다"까지 확인한 셈입니다.

### 오라클 상한이 말해주는 것

경계 두 라벨만 남기고 학습하면 정확도 **0.712**입니다(다수 클래스 기준선 0.527).
경계가 아예 학습되지 않는 것은 아닙니다 — 기준선보다 18%p 높으니 텍스트에 신호가
있기는 합니다.

그런데 캐스케이드는 그 0.712를 못 씁니다. 오라클은 **진짜 경계 건만** 넣어준 값이고,
실제로는 1단계가 잘못 보낸 건이 섞이고 놓친 건은 2단계에 오지도 않습니다. 오차가
곱해져서 전체가 오히려 나빠집니다. 캐스케이드가 0.5957로 가장 낮은 이유입니다.

### 그래서 남는 것

이 경계에서 얻을 수 있는 것은 대략 0.71 근처이고, 그것도 라우팅이 완벽할 때입니다.
라벨 쪽도 이미 확인됐습니다 — 경계 라벨의 재현성이 같은 조건 반복 0.94, 전략 교차
0.85이고, 라벨이 안정적인 82건에서도 모델 정확도는 0.59~0.63입니다.

즉 **모델을 어떻게 짜도, 라벨을 어떻게 고쳐도 이 경계는 여기까지**입니다.
3분류를 유지하는 한 현실적인 천장은 통합 OOF 0.65~0.68입니다. 천장을 올리려면
데이터를 늘리는 수밖에 없습니다.
